# CUDA + Python

## Using Numba for Pythonic GPU-accelerated Code

"**Numba** is a just-in-time compiler for Python functions."

Beyond CPU acceleration, Numba also supports CUDA GPU programming by compiling a **restricted subset of Python code** directly into CUDA kernels and device functions, following the CUDA execution model.

As discussed in class, there are two key points to keep in mind when working with GPU-accelerated code:

1. The data required for kernel execution must be transferred between the host (CPU) and the device (GPU).  
2. GPU kernels provide performance benefits only when the computation effectively utilizes the full bandwidth and parallelism of the GPU.

In this notebook, we will explore how to program a GPU using Python, write CUDA kernels, and manage data exchange between the host and the device.

> **Note on CUDA Backend Compatibility**
>
> In this notebook, we are using **Numba with the built-in CUDA backend**, which is compatible with the older CUDA version (10.2) available on the Jetson platform.  
> 
> In more recent developments, beginning with **CUDA 11**, NVIDIA has taken over maintenance of the CUDA backend for Numba.  
> The same functionalities are now provided through the standalone **`numba-cuda`** package, officially managed and distributed by NVIDIA.
>
> For details and installation instructions for more recent platforms, refer to the official documentation:  
> [https://nvidia.github.io/numba-cuda/](https://nvidia.github.io/numba-cuda/)

We will begin by revisiting the last topic covered in Numba: the `@vectorize` and `@guvectorize` decorators.  
These decorators enable the creation of compiled functions that can target multiple execution environments, including standard `CPU` execution, multithreaded `parallel` execution, and `CUDA` for GPU acceleration.

Before proceeding, let’s use the `numba.cuda` API to detect and list the available CUDA devices.  
This will allow us to verify that the necessary hardware for GPU programming is present and to inspect the device’s capabilities before writing CUDA kernels.

In [ ]:
# Import CUDA and Numba decorators
from numba import cuda, vectorize, guvectorize

We can check whether a CUDA-capable GPU is available on this system by calling the `cuda.is_available()` function.  
This function returns `True` if a compatible GPU and the necessary CUDA drivers are detected, and `False` otherwise.

In [ ]:
# Verify the availability of a CUDA-capable GPU
cuda_available = cuda.is_available()
print(f"{cuda_available=}")

Next, we can detect the available CUDA hardware using the `cuda.detect()` function.  
This function provides a detailed summary of all CUDA-capable devices and their specifications, allowing us to review the hardware configuration and capabilities before proceeding with GPU programming.

In [ ]:
# Display summary information for the available CUDA devices
cuda.detect()

## Writing ufuncs for the GPU with `@vectorize`

Let's redefine a simple `@vectorize` universal function, explicitly specifying that it will be executed as a CUDA kernel on the GPU.

It is important to note that when targeting the GPU, the CUDA versions of the `@vectorize` and `@guvectorize` decorators do **not** behave identically to the NumPy-style `ufunc`s produced by standard Numba compilation.  
These GPU-targeted decorators follow the CUDA execution model, and therefore some differences in behavior and supported operations must be considered when writing GPU-accelerated functions.

In [ ]:
# Define a vectorized universal function for execution on the CPU
@vectorize([
    'int64(int64, int64)',
    'int32(int32, int32)',
    'float64(float64, float64)',
    'float32(float32, float32)'
], target="cpu")
def residual(x, y):
    # Compute the relative difference between two values
    return (y - x) / x

In [ ]:
# Define a vectorized universal function for execution on the GPU
# In this case, both the function signatures and the target type must be specified explicitly
@vectorize([
    'int64(int64, int64)',
    'int32(int32, int32)',
    'float64(float64, float64)',
    'float32(float32, float32)'
], target="cuda")
def residual_cu(x, y):
    # Compute the relative difference between two values
    return (y - x) / x

The inputs and outputs for the `@vectorize` decorator are defined as a list of type signatures, allowing the same universal function (ufunc) to be compiled for multiple data types on the GPU.

A CUDA `ufunc` can operate on arrays that are already located on the GPU device (we will explore this later), but it can also accept arrays that reside on the host.  

Numba automatically handles the transfer of data between the host and the device during the function call, simplifying the process of developing GPU-accelerated code.

In [ ]:
# Import NumPy for array creation
import numpy as np

# Create two random arrays of floating-point values
a = np.random.random(100_000)
b = np.random.random(100_000)

# Check the data type of array 'a'
a.dtype

In [ ]:
%%timeit -n 5 -r 5

# Calculate the residuals using standard NumPy operations
c = (b - a) / a

In [ ]:
%%timeit -n 5 -r 5

# Calculate the residuals using the CPU implementation
c = residual(a, b)

In [ ]:
%%timeit -n 5 -r 5

# Calculate the residuals using the GPU implementation
c = residual_cu(a, b)

In this simple call to the GPU `@vectorize` function, Numba automatically performs several key operations:

- Compiles the CUDA kernel.  
- (!) **Allocates GPU memory** for the input and output arrays.  
- (!) **Copies the input data** from the host (CPU) to the device (GPU).  
- (!) **Executes the CUDA kernel** with the appropriate grid and block dimensions based on the input size.  
- (!) **Copies the results** back from the device (GPU) to the host (CPU).  
- Returns the result as a standard NumPy array residing on the host.

It is a beautifully simple and compact piece of code that performs all the actions we previously executed explicitly in CUDA-C with almost zero effort.

***However,*** the performance may not always meet expectations (i.e. may be crappy).

Based on our experience with CUDA-C, several factors can lead to suboptimal GPU performance:

- **Input Size**:  
  The input arrays may be too small for efficient GPU execution. GPUs are designed for high-throughput workloads rather than low-latency operations, and small data sizes often fail to saturate the device’s compute resources.

- **Computation Complexity**:  
  The performed operation might be too simple. Offloading computations to the GPU incurs overhead compared to executing the same code on the CPU.  
  If the ratio of *Compute operations to Global Memory Accesses* (CGMA) is too low, global memory latency can dominate runtime and negate any parallelization benefits.

- **Data Transfer Overhead**:  
  The `timeit` measurement includes data transfer between the host (CPU) and the device (GPU).  
  Although Numba handles these transfers automatically, repeatedly copying data for each function call is inefficient.  
  A better strategy is to transfer data to the GPU once, perform multiple operations on-device, and copy results back only when necessary.

- **Data Types**:  
  The data types used may not be optimal for the computation.  
  For instance, using `float64` instead of `float32` doubles both memory footprint and transfer time.  
  When numerical precision requirements allow, smaller data types can significantly improve performance.

As we have discussed, achieving optimal GPU performance requires more than simply dispatching existing code to the device.  
It involves rethinking algorithmic structure, analyzing data size relative to computational complexity, and carefully managing memory transfers to fully exploit the GPU’s capabilities.

In [ ]:
import math

# Precompute this constant as a float32
SQRT_2PI = np.float32((2 * math.pi) ** 0.5)

# Define a vectorized function to compute Gaussian values on the GPU
@vectorize(['float32(float32, float32, float32)'], target="cuda")
def gaussian_cu(x, m, s):
    # Compute the Gaussian probability density function (PDF)
    return math.exp(-0.5 * ((x - m) / s) ** 2) / (s * SQRT_2PI)

# Define a NumPy equivalent for comparison on the CPU
def gaussian_np(x, m, s):
    return np.exp(-0.5 * ((x - m) / s) ** 2) / (s * SQRT_2PI)

In [ ]:
# Prepare data to evaluate the Gaussian function a few million times
# All evaluations use the same mean and standard deviation
x = np.random.uniform(-5, 5, size=10_000_000).astype("float32")
mean = np.float32(0.0)
sigma = np.float32(1.0)

In [ ]:
%%timeit -n 5 -r 5

# Compute the Gaussian values using the NumPy implementation
gaussian_np(x, mean, sigma)

In [ ]:
%%timeit -n 5 -r 5

# Compute the Gaussian values using the Numba+CUDA implementation
gaussian_cu(x, mean, sigma)

As expected, we observe performance improvements resulting from several key optimizations:

- **Using `float32` Data Type**:  
  Leveraging `float32` instead of `float64` reduces memory consumption and improves computation speed.

- **Increased Computation per Memory Access**:  
  The GPU function performs a higher ratio of computation relative to memory access, which is essential for maximizing GPU performance.

- **High Concurrency**:  
  A large number of GPU threads execute concurrently, increasing throughput and overall efficiency.

It is important to note that the measured execution time for the GPU function still includes the overhead of transferring data between the host (CPU) and the device (GPU).  
This data transfer cost can significantly affect overall performance, especially for smaller workloads where the computation time is relatively short.

> **Why are we using `math.exp` instead of `np.exp`?**  
>
> We use `math.exp` instead of `np.exp` because of the current limitations of Numba’s CUDA support for NumPy functions.  
> Depending on the installed version of Numba, certain NumPy mathematical functions may not yet be supported for execution on the GPU.  
> As a result, scalar functions from Python’s built-in `math` module are often required when targeting CUDA.
>
> While `np.exp` (and other NumPy functions) are gradually gaining support in newer releases of Numba, the version we are using still requires this approach.
>
> Given our current hardware and software configuration:
>
> ```bash
> # Name                    Version                   Build  Channel
> python                    3.9.10       h1b383ca_2_cpython    conda-forge
> numpy                     1.23.5           py39hf5a3166_0    conda-forge
> numba                     0.56.4           py39h6619693_1    conda-forge
> ```
>
> Recent versions of Numba have significantly expanded compatibility with NumPy functions on CUDA.  
> As always, consulting the documentation for your specific version is essential:  
> [https://numba.readthedocs.io/en/stable/cuda/cudapysupported.html](https://numba.readthedocs.io/en/stable/cuda/cudapysupported.html)  (for numba)
> [https://nvidia.github.io/numba-cuda/user/cudapysupported.html](https://nvidia.github.io/numba-cuda/user/cudapysupported.html) (for numba-cuda)

## Writing ufuncs for the GPU with `@guvectorize`

The same considerations discussed earlier for `@vectorize` also apply to functions that operate on entire arrays using `@guvectorize`.

As an example, we can rewrite the matrix element-wise addition code using Numba and CUDA with the `@guvectorize` decorator.

In [ ]:
# Define the signature and array layouts for the guvectorize function
# Two input matrices (A, B) and one output matrix (C)
# All elements are of type float32
@guvectorize(
    ["(float32[:, :], float32[:, :], float32[:, :])"], # No return signature. `void` can be used to be more explicit.
    "(x,y),(x,y)->(x,y)",
    target="cuda"
)
def matrix_addition(A, B, C):
    # Loop over each element of the 2D input arrays
    for i in range(A.shape[0]):
        for j in range(A.shape[1]):
            # Perform element-wise addition
            C[i, j] = A[i, j] + B[i, j]

In [ ]:
# Define the size of the matrices
rows, cols = 1024, 1024

# Create two random matrices A and B with float32 elements
A = np.random.rand(rows, cols).astype(np.float32)
B = np.random.rand(rows, cols).astype(np.float32)

# Initialize an empty matrix for the result C
C = np.zeros((rows, cols), dtype=np.float32)

In [ ]:
%%timeit -n 1 -r 1

# Perform matrix addition using the guvectorized GPU function
matrix_addition(A, B, C)

In [ ]:
# Display the resulting matrix
C

> Remember that `@guvectorize` does not allow returning the result directly.   
> Instead, the result object must be passed as an input parameter. This design choice emphasizes the need for explicit memory management in GPU programming, ensuring that the outputs are written to a predefined location in memory.

## Memory Management

So far, we have operated directly on NumPy arrays located on the host.  

During kernel execution, Numba automatically transfers these arrays to the device and copies the results back to the host after execution.  
While this automatic behavior is convenient, it is not particularly efficient for workloads involving multiple GPU operations.

In most cases, it is preferable to keep data on the GPU and launch several kernels sequentially, avoiding unnecessary data transfers between the device and the host.

We can use the CUDA APIs provided by Numba to manage GPU memory manually, allowing for more efficient data handling:

- **`cuda.device_array`**: Allocates memory on the device.  
- **`cuda.to_device`**: Allocates memory on the device and, by default (`copy=True`), copies data from an existing host array.  
- **`cuda.copy_to_host`**: Transfers data from device memory back to the host.

Let’s return to the previous example and apply these memory management concepts.

In [ ]:
# Define the size of the matrices
rows, cols = 1024, 1024

# Create host arrays A and B with random float32 values
A = np.random.rand(rows, cols).astype(np.float32)
B = np.random.rand(rows, cols).astype(np.float32)

In [ ]:
# Check whether A is a CUDA array (i.e., managed by the CUDA backend)
cuda.is_cuda_array(A)

In [ ]:
# Create device arrays and perform a host-to-device copy
# Both allocation and copying are handled by cuda.to_device()
d_A = cuda.to_device(A)
d_B = cuda.to_device(B)

# Allocate a device array with the same shape and data type as d_A
d_C = cuda.device_array_like(d_A)

In [ ]:
# Check if d_A is a CUDA array
cuda.is_cuda_array(d_A)

In [ ]:
# Print the shape and memory allocation details for the device array d_A
print(f'Shape of array d_A in device memory: {d_A.shape}')
print(f'Bytes allocated for d_A: {d_A.nbytes / 1e6:.1f} MB')

In [ ]:
%%timeit -n 3 -r 3

# Perform matrix addition using the guvectorized GPU function with device memory
matrix_addition(d_A, d_B, d_C)

Note that this time measurement is now **completely bogus**!

Launching a CUDA kernel is an **asynchronous** operation, meaning it does not block the CPU while the GPU executes the task.  
As a result, the timing reported by `%%timeit` does not reflect the true execution time of the computation on the GPU.

To obtain reliable performance measurements, we should instead use **CUDA Events**.  
CUDA Events allow us to record timestamps immediately before and after kernel execution, providing precise measurements of the actual GPU execution time.

In [ ]:
# Create CUDA events for timing
start = cuda.event()
stop = cuda.event()

# Start the timer
start.record()

# Perform matrix addition using device memory
matrix_addition(d_A, d_B, d_C)

# Stop the timer and synchronize
stop.record()
stop.synchronize()

# Get the elapsed time between start and stop (in milliseconds)
elapsed_time = cuda.event_elapsed_time(start, stop)
print(f"Elapsed time: {elapsed_time:.1f} ms")

In [ ]:
# Copy the result from the device back to the host
C = d_C.copy_to_host()

# Display the resulting matrix
C

We have discussed the importance of freeing device memory after kernel execution to avoid unnecessary GPU memory consumption.  

However, it is important to remember that **Python is not C or C++**.  
An explicit call analogous to `cudaFree()` will not have an immediate effect when issued.  

Python is a **garbage-collected** language, meaning that memory deallocation is handled automatically by the interpreter at an indeterminate future time. We know it will be freed, but only eventually...  

While it remains good practice to deallocate GPU memory objects when they are no longer needed, Python’s memory management model introduces additional complexity. 
We must therefore be mindful of how and when we release device memory to prevent potential memory leaks or excessive memory usage.

In [ ]:
# Delete (free?) the device-allocated arrays
del d_A, d_B, d_C

When this command is executed, Python removes the references to the `d_A`, `d_B`, and `d_C` objects, but it does **not** immediately free the GPU memory they occupy.  

Numba will release this memory only when Python’s garbage collector runs, which may not occur right away.  

We do not need to understand all the underlying implementation details, but it is important to recognize that this behavior illustrates one key difference between Python and memory-managed languages such as `C` or `C++`.  
In those languages, developers have explicit control over memory allocation and deallocation, resulting in more predictable and immediate memory management.

## Expressing CUDA Kernels with `@jit`

In addition to creating GPU kernels with the `@vectorize` decorator, Numba allows you to define **custom CUDA kernels** using `@jit` or `@njit`.  
These decorators let you express full functions targeted at the GPU rather than the CPU.

Let’s revisit the Julia fractal example and rewrite the Numba implementation to execute as a CUDA kernel.

To do this, we need to determine the location of each thread within the grid during execution.  
The `numba.cuda` module provides access to these indices, mirroring what we use in `CUDA-C`.

For a **1D grid** of blocks and threads, you can compute the unique index for each thread as:  
```python
tx = cuda.threadIdx.x
bx = cuda.blockIdx.x
bw = cuda.blockDim.x
idx = tx + bx * bw
```

This allows you to access each data element using:  
```python
data[idx] = ...
```

Similarly, for a 2D grid, the indices can be computed as:  
```python
tx = cuda.threadIdx.x
ty = cuda.threadIdx.y

bx = cuda.blockIdx.x
by = cuda.blockIdx.y

bw = cuda.blockDim.x
bh = cuda.blockDim.y

idx_x = tx + bx * bw
idx_y = ty + by * bh

data[idx_x, idx_y] = ...
```

Alternatively, `numba.cuda` provides a more convenient API through the function `cuda.grid(ndim)`, which returns the unique global index of the current thread in either 1D or 2D form:
```python
idx = cuda.grid(1)
```

or
```python
idx_x, idx_y = cuda.grid(2)
```

These functions return the _absolute position_ of the current thread within the full grid of blocks.
The argument `ndim` must match the number of dimensions specified when launching the kernel.
If `ndim = 1`, a single integer is returned; if `ndim = 2` or `3`, a tuple of integers is returned.

Similarly, the function `cuda.gridsize()` returns the total number of threads across the grid, i.e. the shape in threads of the entire grid of blocks.
This provides a way to query the overall dimensions of the grid, using the same `ndim` parameter as above.

### Re-computing the Julia Set with `cuda.jit`

We will now rewrite the previous Numba implementation of the Julia set to run directly on the GPU using the `cuda.jit` decorator.  

This approach allows us to express the kernel explicitly, managing thread indexing and memory access in a way that mirrors standard CUDA programming.

In [ ]:
@cuda.jit
def julia_fractal(z_re, z_im, j):
    # Get the position of the thread in the overall grid
    idx_x, idx_y = cuda.grid(2)

    """
    Equivalent explicit indexing:
        tx = cuda.threadIdx.x
        bx = cuda.blockIdx.x
        bw = cuda.blockDim.x
        idx_x = tx + bx * bw

        ty = cuda.threadIdx.y
        by = cuda.blockIdx.y
        bh = cuda.blockDim.y
        idx_y = ty + by * bh
    """

    # Ensure the thread index is within the bounds of the output array
    if idx_x < j.shape[0] and idx_y < j.shape[1]:
        # Initialize the complex number z for this coordinate
        z = z_re[idx_x] + 1j * z_im[idx_y]

        # Perform iterations to check for divergence
        for t in range(256):
            z = z * z - 0.05 + 0.68j
            if (z.real * z.real + z.imag * z.imag) > 4.0:
                j[idx_x, idx_y] = t
                break

In [ ]:
# Define grid dimensions; in this case, a square grid of size N x N
N = 1024
width, height = np.int32(N), np.int32(N)

# Create arrays for the real and imaginary components of the complex plane
z_real = np.linspace(-1.5, 1.5, width).astype("float32")
z_imag = np.linspace(-1.5, 1.5, height).astype("float32")

# Prepare the output array
# Use uint8 since we only need to store iteration counts between 0 and 255
j = np.zeros((width, height), dtype=np.uint8)

In [ ]:
# Create device arrays and perform host-to-device copies
# Both allocation and data transfer are handled by cuda.to_device()
d_z_real = cuda.to_device(z_real)
d_z_imag = cuda.to_device(z_imag)
d_j = cuda.to_device(j)

In [ ]:
# Define the CUDA grid dimensions
threads_per_block = (8, 8)  # 8x8 threads per block

# Compute the number of blocks required in each dimension
blocks_per_grid_x = math.ceil(j.shape[0] / threads_per_block[0])
blocks_per_grid_y = math.ceil(j.shape[1] / threads_per_block[1])

# Alternatively:
# blocks_per_grid_x = (j.shape[0] + threads_per_block[0] - 1) // threads_per_block[0]
# blocks_per_grid_y = (j.shape[1] + threads_per_block[1] - 1) // threads_per_block[1]

blocks_per_grid = (blocks_per_grid_x, blocks_per_grid_y)

In [ ]:
# Create CUDA events for precise timing
start = cuda.event()
stop = cuda.event()

# Start the timer
start.record()

# Launch the CUDA kernel
julia_fractal[blocks_per_grid, threads_per_block](d_z_real, d_z_imag, d_j)

# Stop the timer and synchronize
stop.record()
stop.synchronize()

# Compute and display the elapsed time (in milliseconds)
elapsed_time = cuda.event_elapsed_time(start, stop)
print(f"Elapsed time: {elapsed_time:.1f} ms")

In [ ]:
# Copy the result back to the host, storing it in the existing array j
d_j.copy_to_host(j)

In [ ]:
import matplotlib.pyplot as plt

# Display the Julia set
fig, ax = plt.subplots(figsize=(12, 12))
ax.imshow(j, cmap=plt.cm.RdBu_r, extent=[-1.5, 1.5, -1.5, 1.5])

# Axis labels for the complex plane
ax.set_xlabel(r"$\mathrm{Re}(z)$", fontsize=18)
ax.set_ylabel(r"$\mathrm{Im}(z)$", fontsize=18)

plt.show();

In [ ]:
# Attempt to free GPU memory by deleting device arrays
del d_z_real, d_z_imag, d_j

### Shared Memory and Thread Synchronization in Numba CUDA Kernels

As discussed in CUDA-C, optimizing CUDA kernel performance often involves **reducing latency associated with global memory access** by leveraging **shared memory**.  
Shared memory resides locally within each Streaming Multiprocessor (SMPs) and can be accessed by all threads within the same block, providing much faster access compared to global memory.

Numba supports the use of shared memory in CUDA kernels through the function:  
```python
cuda.shared.array(shape, dtype)
```

This call must be placed inside a `cuda.jit` kernel and is typically used in situations where memory coalescing or data reuse among threads can yield performance improvements.

To ensure that all threads within a block have completed their operations on shared memory before proceeding, it is necessary to include synchronization points using:  
```python
cuda.syncthreads()
```

### Square Matrix Multiplication with Tiling

We now translate the classic **CUDA-C tiled square matrix multiplication** example into Python using **Numba+CUDA**.  

For consistency and easier comparison with the original `CUDA-C` implementation, we will limit this version to **square matrices** whose width is a multiple of the tile size.

Below we can refer to the `CUDA-C` implementation for reference:


```c
#define WIDTH 2048                      
#define TILE_WIDTH 32                   
#define THREADS_PER_BLOCK_X TILE_WIDTH  
#define THREADS_PER_BLOCK_Y TILE_WIDTH  

[...]


__global__ void matrixMultiplication(const float* M, const float* N, float* P, const int width) {
    __shared__ float M_tile[TILE_WIDTH][TILE_WIDTH];
    __shared__ float N_tile[TILE_WIDTH][TILE_WIDTH];
    
    int tx = threadIdx.x;
    int ty = threadIdx.y;

    int row = blockIdx.y * TILE_WIDTH + ty;
    int col = blockIdx.x * TILE_WIDTH + tx;

    float sum = 0.;

    // Fill the shared memory
    // Loop over the tiles of the input matrices
    for (int t = 0; t < width / TILE_WIDTH; ++t) {
        if ( (row < width) && (t * TILE_WIDTH + tx < width) )
            M_tile[ty][tx] = M[row * width + t * TILE_WIDTH + tx];
        else 
            M_tile[ty][tx] = 0.;

        if ( (t * TILE_WIDTH + ty < width) && (col < width) )
            N_tile[ty][tx] = N[(t * TILE_WIDTH + ty) * width + col];
        else 
            N_tile[ty][tx] = 0.;

        // Synchronize (ensure the tile is loaded in shared memory)
        __syncthreads();

    
        // Perform the multiplication for this tile
        for (int k = 0; k < TILE_WIDTH; ++k) {
            sum += M_tile[ty][k] * N_tile[k][tx];
        }

        // Ensure all threads are done computing before loading the next tile
        __syncthreads(); 
    }

    // Write the result back to the global memory
    if (row < width && col < width) {
        P[row * width + col] = sum;
    }
}
```

In [ ]:
# Define constants for matrix dimensions and tile size
WIDTH = 2048                 
TILE_WIDTH = 32                   

# Set number of threads per block in both dimensions
THREADS_PER_BLOCK_X = TILE_WIDTH  
THREADS_PER_BLOCK_Y = TILE_WIDTH  

@cuda.jit
def matrix_multiplication(M, N, P):
    # Allocate shared memory for tiles
    M_tile = cuda.shared.array(shape=(TILE_WIDTH, TILE_WIDTH), dtype='float32')
    N_tile = cuda.shared.array(shape=(TILE_WIDTH, TILE_WIDTH), dtype='float32')

    # Thread index within the block
    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y

    # Calculate the row and column indices for the global matrix
    row = cuda.blockIdx.y * TILE_WIDTH + ty
    col = cuda.blockIdx.x * TILE_WIDTH + tx

    # Initialize sum for the resulting matrix element
    sum = np.float32(0.)

    # Loop over the tiles of the input matrices
    for t in range(M.shape[0] // TILE_WIDTH):
        # Load tile from matrix M into shared memory
        if (row < WIDTH) and (t * TILE_WIDTH + tx < WIDTH):
            M_tile[ty][tx] = M[row][t * TILE_WIDTH + tx]
        else:
            M_tile[ty][tx] = 0.

        # Load tile from matrix N into shared memory
        if (t * TILE_WIDTH + ty < WIDTH) and (col < WIDTH):
            N_tile[ty][tx] = N[t * TILE_WIDTH + ty][col]
        else:
            N_tile[ty][tx] = 0.

        # Synchronize threads to ensure all data is loaded
        cuda.syncthreads()

        # Compute partial sum for the current tile
        for k in range(TILE_WIDTH):
            sum += M_tile[ty][k] * N_tile[k][tx]
        
        # Synchronize threads before loading the next tile
        cuda.syncthreads()
    
    # Write the result to the output matrix P
    if (row < WIDTH) and (col < WIDTH):
        P[row][col] = sum

In [ ]:
# Define the input and output matrices
A = np.random.random(size=(WIDTH, WIDTH)).astype("float32")
B = np.random.random(size=(WIDTH, WIDTH)).astype("float32")

# Initialize the output matrix C with zeros (same shape and type as A)
C = np.zeros_like(A)

In [ ]:
# Allocate and transfer data to the device
d_A = cuda.to_device(A)  # Copy input matrix A to device memory
d_B = cuda.to_device(B)  # Copy input matrix B to device memory

# Allocate device array for the output matrix C (same shape and dtype as A)
d_C = cuda.device_array_like(d_A)

In [ ]:
# Define the CUDA grid size
threads_per_block = (THREADS_PER_BLOCK_X, THREADS_PER_BLOCK_Y)  # Threads per block (x, y)

# Compute the number of blocks required in each dimension
blocks_per_grid_x = math.ceil(d_C.shape[0] / threads_per_block[0])
blocks_per_grid_y = math.ceil(d_C.shape[1] / threads_per_block[1])

# Combine into a single tuple for grid configuration
blocks_per_grid = (blocks_per_grid_x, blocks_per_grid_y)

In [ ]:
# Create CUDA events to measure kernel execution time
start = cuda.event()  # Start event
stop = cuda.event()   # Stop event

# Record the start time
start.record()

# Launch the CUDA kernel for matrix multiplication
matrix_multiplication[blocks_per_grid, threads_per_block](d_A, d_B, d_C)

# Record the stop time and synchronize
stop.record()
stop.synchronize()  # Wait for the kernel to finish

# Compute and display the elapsed time (in milliseconds)
elapsed_time = cuda.event_elapsed_time(start, stop)
print(f"Elapsed time: {elapsed_time:.1f} ms")

In [ ]:
# Copy the result matrix from device to host
C = d_C.copy_to_host()  

# Display the resulting matrix
C

> Despite the optimizations implemented so far, it is important to recognize that with relatively small datasets, the CUDA implementation may not achieve significant speedups compared to the straightforward NumPy version.
> 
> CUDA delivers its greatest performance advantages on **large-scale computations**, where the overhead associated with kernel launches and data transfers is amortized over a substantial workload.  
> For smaller matrices, however, this overhead can dominate execution time, making the parallelization less effective or even slower than CPU-based solutions.

In [ ]:
# Create an empty NumPy array for the output (same shape and dtype as A)
C_np = np.empty_like(A)

In [ ]:
%%timeit -n 3 -r 3  

# Perform matrix multiplication using NumPy's matmul function
np.matmul(A, B, C_np)

In [ ]:
# Verify that the GPU and NumPy results are nearly identical
np.allclose(C, C_np)

In [ ]:
# Mark device memory for deletion
del d_A, d_B, d_C